In [44]:
!pip install -r requirements.txt

# Классификация ботов

Модель: CatBoost на поведенческих признаках куки и признаках по просмотренным объявлениям.
Валидация по времени, как в quickstart.

| модель | P@R≥0.7 |
|---|---|
| baseline из quickstart | 0.096 |
| CatBoost | 0.869 |

In [45]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from metric import precision_at_recall

RANDOM_STATE = 11
np.random.seed(RANDOM_STATE)

## Данные

In [46]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])
cookies = pd.concat([train.drop(columns='target'), test], ignore_index=True) # все куки train + test

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


## Проверка событий

In [47]:
ev_all = events.merge(pd.concat([train.assign(part='train'), test.assign(part='test')]), on='cookie_id')
ev_all['in_window'] = (ev_all.event_ts >= ev_all.window_start_ts) & (ev_all.event_ts < ev_all.window_end_ts)

print('доля событий в окне:', ev_all.groupby('part').in_window.mean().round(3).to_dict())
print('дубликатов строк:', events.duplicated().sum())
print('события отсортированы:', events.event_ts.is_monotonic_increasing)
print('значения platform:', events.platform.unique())

доля событий в окне: {'test': 1.0, 'train': 0.83}
дубликатов строк: 4863
события отсортированы: False
значения platform: <ArrowStringArray>
['desktop',     'web',     'WEB',     'Web', 'Android',     'IOS', 'ANDROID',
 'android',     'ios',     'iOS',  'iphone']
Length: 11, dtype: str


В train есть события после окна. В test таких нет. Из условия будем работать только с событиями в окне.
Дубликаты удаляем, события сортируем по id куки и времени.
platform приводим к одному из видов web, android, ios.

## Подготовка событий

In [48]:
PLATFORM_NORM = {'web': 'web', 'desktop': 'web', 'android': 'android', 'ios': 'ios', 'iphone': 'ios'}

def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

def clean_events(events, meta):
    ev = events_in_window(events.drop_duplicates(), meta).copy()
    ev['platform'] = ev.platform.str.lower().str.strip().map(PLATFORM_NORM)
    ev['day'] = ev.window_start_ts
    return ev.sort_values(['cookie_id', 'event_ts', 'eid']).reset_index(drop=True)


ev = clean_events(events, cookies)
print(len(ev), 'событий,', len(ev.cookie_id.unique()), 'кук')

283833 событий, 16000 кук


## Признаки

### Какие лучше выбрать?

Предпологаем, что бот отличается от человека тем, как и что он смотрит. Поэтому смотрим на признаки поведения внутри окна. Скрипт действует через ровные короткие интервалы, а человек делает паузы. Также бот почти не листает фото и не добавляет в избранное. Ну и курсор у человека гуляет по всему экрану, в отличие от бота

Итого:

ритм: интервалы между событиями, сессии, событий в минуту; состав действий: доли типов событий; разнообразие: категории, города, запросы, номера страниц в поиске; курсор: разброс и шаг, только web; User-Agent: класс клиента и версия; возраст куки

In [49]:
EVENT_TYPES = ['search_results_view', 'item_view', 'photo_swipe', 'seller_page_view',
               'contact_phone_show', 'contact_chat_open', 'contact_message_sent',
               'favorite_add', 'login']
CONTACT_EVENTS = ['contact_phone_show', 'contact_chat_open', 'contact_message_sent']
SESSION_TIMEOUT = 1800 # пауза больше 30 мин = новая сессия
GAP_BINS = [0, 1, 3, 6, 12, 20, 30, 60, 180, SESSION_TIMEOUT]
SHARE_EVENTS = ['search_results_view', 'item_view', 'photo_swipe', 'seller_page_view',
                'contact_phone_show', 'contact_chat_open', 'contact_message_sent']
COUNT_EVENTS = ['search_results_view', 'item_view', 'photo_swipe', 'seller_page_view',
                'contact_phone_show', 'favorite_add', 'login']
GAP_BIN_NAMES = ['dtbin_0_1', 'dtbin_1_3', 'dtbin_6_12', 'dtbin_12_20', 'dtbin_20_30',
                 'dtbin_30_60', 'dtbin_60_180', 'dtbin_180_1800']


def client_type(agent):
    """Класс клиента по строке User-Agent. Группируем сырые строки в 7 классов:
    headless, app, http_lib, yabrowser, firefox, mobile_browser, chrome."""
    if 'HeadlessChrome' in agent:
        return 'headless'
    if agent.startswith('Avito/'):
        return 'app'
    if not agent.startswith('Mozilla'):
        return 'http_lib'
    if 'YaBrowser' in agent:
        return 'yabrowser'
    if 'Firefox' in agent:
        return 'firefox'
    if 'iPhone' in agent or 'Android' in agent:
        return 'mobile_browser'
    return 'chrome'


def spread_stats(df, col):
    """Энтропия и доля самого частого значения col у каждой куки."""
    counts = df.dropna(subset=[col]).groupby(['cookie_id', col]).size()
    share = counts / counts.groupby(level=0).transform('sum')
    return -(share * np.log(share)).groupby(level=0).sum(), share.groupby(level=0).max()


def build_features(ev, meta):
    ev = ev.copy()
    ev['hour'] = ev.event_ts.dt.hour
    ev['night'] = ev.hour < 6
    ev['minute'] = ev.event_ts.dt.floor('min')
    ev['dt'] = ev.groupby('cookie_id', sort=False).event_ts.diff().dt.total_seconds()
    ev['dt_in'] = ev.dt.where(ev.dt <= SESSION_TIMEOUT)
    ev['session_id'] = (ev.dt.isna() | (ev.dt > SESSION_TIMEOUT)).astype(int).groupby(ev.cookie_id).cumsum()
    by_cookie = ev.groupby('cookie_id', sort=False)

    info = meta.set_index('cookie_id')
    feats = pd.DataFrame(index=pd.Index(meta.cookie_id.values, name='cookie_id'))
    t_first, t_last = by_cookie.event_ts.min(), by_cookie.event_ts.max()

    # возраст куки
    feats['first_since_created_h'] = (t_first.reindex(feats.index) - info.cookie_created_at.reindex(feats.index)).dt.total_seconds() / 3600   # часов от создания до 1-го события
    feats['cookie_age_h'] = (info.window_start_ts - info.cookie_created_at).dt.total_seconds().reindex(feats.index) / 3600   # часов от создания до окна

    # платформа и User-Agent: самое частое значение у куки
    most_common = lambda col: ev.groupby(['cookie_id', col]).size().reset_index().sort_values(0, kind='mergesort').drop_duplicates('cookie_id', keep='last').set_index('cookie_id')[col]
    agent = most_common('user_agent')
    feats['n_ua'] = by_cookie.user_agent.nunique()
    feats['ua_major'] = agent.str.extract(r'(?:Chrome|Firefox|Avito|Version)/(\d+)')[0].astype(float)   # версия браузера
    feats['ua_class'] = agent.map(client_type) # класс клиента
    feats['platform'] = most_common('platform') # web, android, ios

    # курсор
    cursor = ev[ev.pointer_x.notna()].copy()
    by_cur = cursor.groupby('cookie_id')
    cursor['step'] = np.hypot(by_cur.pointer_x.diff(), by_cur.pointer_y.diff())
    cursor['edge'] = (cursor.pointer_x <= 0) | (cursor.pointer_x >= 1919) | (cursor.pointer_y <= 0) | (cursor.pointer_y >= 1079)
    feats['ptr_step_mean'] = cursor.groupby('cookie_id').step.mean()
    feats['ptr_edge_share'] = cursor.groupby('cookie_id').edge.mean()
    feats['ptr_x_std'], feats['ptr_y_std'] = by_cur.pointer_x.std(), by_cur.pointer_y.std()
    feats['ptr_x_range'] = by_cur.pointer_x.max() - by_cur.pointer_x.min()
    feats['ptr_y_range'] = by_cur.pointer_y.max() - by_cur.pointer_y.min()
    feats['ptr_x_mean'], feats['ptr_y_mean'] = by_cur.pointer_x.mean(), by_cur.pointer_y.mean()

    # состав действий
    feats['n_events'] = by_cookie.size()
    counts = pd.crosstab(ev.cookie_id, ev.event_name).reindex(columns=EVENT_TYPES, fill_value=0)
    for etype in SHARE_EVENTS:
        feats[f'share_{etype}'] = counts[etype] / feats.n_events
    for etype in COUNT_EVENTS:
        feats[f'cnt_{etype}'] = counts[etype]
    n_views = counts['item_view'].replace(0, np.nan)
    feats['search_per_view'] = counts['search_results_view'] / n_views
    feats['seller_per_view'] = counts['seller_page_view'] / n_views
    feats['photo_per_view'] = counts['photo_swipe'] / n_views
    feats['fav_per_view'] = counts['favorite_add'] / n_views
    feats['contact_per_view'] = counts[CONTACT_EVENTS].sum(axis=1) / n_views
    ev['same_as_prev'] = ev.event_name.eq(by_cookie.event_name.shift())
    feats['same_type_repeat_share'] = ev.groupby('cookie_id').same_as_prev.mean()

    # поиск
    search = ev[ev.event_name == 'search_results_view'][['cookie_id', 'search_query', 'search_page', 'item_location']].copy()
    search['pos'] = np.arange(len(search))
    feats['loc_per_query_max'] = search.groupby(['cookie_id', 'search_query']).item_location.nunique().groupby('cookie_id').max()
    # прыжок: открыта страница p > 1, а страница p-1 того же запроса раньше не открывалась
    prev_page = search.groupby(['cookie_id', 'search_query', 'search_page']).pos.min().rename('prev_pos').reset_index()
    prev_page['search_page'] += 1
    search = search.merge(prev_page, on=['cookie_id', 'search_query', 'search_page'], how='left')
    search['jump'] = (search.search_page > 1) & ~(search.prev_pos < search.pos)
    feats['page_jump_share'] = search.groupby('cookie_id').jump.mean()
    feats['page1_share'] = (search.search_page == 1).groupby(search.cookie_id).mean()
    feats['page_ge5_share'] = (search.search_page >= 5).groupby(search.cookie_id).mean()
    feats['page_mean'] = search.groupby('cookie_id').search_page.mean()

    # разнообразие
    feats['n_query'] = by_cookie.search_query.nunique().where(lambda x: x > 0)
    feats['query_entropy'], feats['query_top_share'] = spread_stats(ev, 'search_query') # разнообразие и доля самого частого
    feats['loc_entropy'], feats['loc_top_share'] = spread_stats(ev, 'item_location')
    feats['n_cat'] = by_cookie.item_category.nunique().where(lambda x: x > 0)
    feats['cat_entropy'], feats['cat_top_share'] = spread_stats(ev, 'item_category')
    n_cities = by_cookie.item_location.nunique().where(lambda x: x > 0)
    feats['loc_per_cat'] = n_cities / feats.n_cat   # обход одного раздела по многим городам
    feats['loc_per_event'] = n_cities / feats.n_events
    feats['cat_na_share'] = ev.item_category.isna().groupby(ev.cookie_id).mean()
    item_ev = ev[ev.item_id.notna()]
    by_item = item_ev.groupby('cookie_id')
    feats['seller_pro_share'] = (item_ev.seller_type == 'pro').groupby(item_ev.cookie_id).mean()
    feats['n_items'] = by_item.item_id.nunique()
    feats['item_repeat_share'] = 1 - feats.n_items / by_item.size()

    # ритм: сессии
    sessions = ev.groupby(['cookie_id', 'session_id']).agg(n=('eid', 'size'), t0=('event_ts', 'min'), t1=('event_ts', 'max'))
    sessions['dur_min'] = (sessions.t1 - sessions.t0).dt.total_seconds() / 60
    by_sess = sessions.groupby('cookie_id')
    feats['max_session_rate'] = (sessions.n / (sessions.dur_min + 1)).groupby('cookie_id').max() # +1, чтобы не делить на 0
    feats['mean_session_dur_min'] = by_sess.dur_min.mean()
    feats['max_session_events'] = by_sess.n.max()
    feats['events_per_session'] = by_sess.n.mean()

    # ритм: интервалы внутри сессии
    gaps_in = by_cookie.dt_in
    gap_bin = pd.cut(ev.dt_in, [-1] + GAP_BINS[1:], labels=[f'dtbin_{a}_{b}' for a, b in zip(GAP_BINS[:-1], GAP_BINS[1:])])
    bin_shares = pd.crosstab(ev.cookie_id, gap_bin, normalize='index').rename(columns=str) # доли интервалов по бинам
    feats = feats.join(bin_shares[GAP_BIN_NAMES])
    ev['log_dtin'] = np.log1p(ev.dt_in)
    feats['log_dtin_std'] = ev.groupby('cookie_id').log_dtin.std()
    feats['dtin_median'] = gaps_in.median()
    ev['dtin_absdev'] = (ev.dt_in - gaps_in.transform('median')).abs()
    feats['dtin_mad_rel'] = ev.groupby('cookie_id').dtin_absdev.median() / (feats.dtin_median + 1) # чем ближе признак к нулю, тем ровнее интервалы
    feats['dtin_std'] = gaps_in.std()
    feats['dtin_mean'] = gaps_in.mean()
    feats['dtin_cv'] = feats.dtin_std / feats.dtin_mean

    # ритм: все интервалы и время активности
    gaps = by_cookie.dt
    feats['dt_std'] = gaps.std()
    feats['dt_mean'] = gaps.mean()
    feats['dt_cv'] = feats.dt_std / feats.dt_mean
    for q in [0.75, 0.5, 0.25, 0.1]:
        feats[f'dt_q{int(q * 100)}'] = gaps.quantile(q)
    feats['hour_entropy'], _ = spread_stats(ev, 'hour') # разброс активности по часам
    feats['night_share'] = by_cookie.night.mean()
    feats['n_hours'] = by_cookie.hour.nunique()
    feats['n_active_minutes'] = by_cookie.minute.nunique()
    feats['events_per_minute'] = feats.n_events / feats.n_active_minutes
    feats['first_event_h'] = (t_first - info.window_start_ts.reindex(t_first.index)).dt.total_seconds() / 3600 # час первого события
    feats['span_h'] = (t_last - t_first).dt.total_seconds() / 3600
    return feats


features = build_features(ev, cookies)
features.shape

(16000, 86)

## Связь с ботами: объявления, города, User-Agent

Проверяем для каждого поля один и тот же тест, а именно по первой неделе train считаем долю ботов для каждого значения,
потом для кук 2ой недели проверяем, насколько эта доля отличает ботов от людей

In [50]:
ev_lab = ev.merge(train[['cookie_id', 'target']], on='cookie_id')
week1_ev, week2_ev = ev_lab[ev_lab.day < '2026-04-13'], ev_lab[ev_lab.day >= '2026-04-13']
base_rate = train.target.mean()


def key_auc(keys):
    known = week1_ev.dropna(subset=keys).drop_duplicates(['cookie_id'] + keys).groupby(keys).target.agg(['sum', 'count'])
    rate = ((known['sum'] + base_rate * 2) / (known['count'] + 2)).rename('rate') # сглаженная доля ботов
    rows = week2_ev.dropna(subset=keys).join(rate, on=keys)
    rows['rate'] = rows.rate.fillna(base_rate)
    per_cookie = rows.groupby('cookie_id').agg(rate=('rate', 'mean'), target=('target', 'first'))
    return round(roc_auc_score(per_cookie.target, per_cookie.rate), 3)

### Объявления и боты

In [51]:
item_views = ev.loc[ev.item_id.notna(), ['cookie_id', 'item_id', 'day']].drop_duplicates().merge(train[['cookie_id', 'target']], on='cookie_id')

viewers = item_views.groupby('item_id').cookie_id.nunique()
print('зрителей у просмотренного объявления, в среднем:',
      item_views.assign(v=item_views.item_id.map(viewers)).groupby('target').v.mean().round(1).to_dict())

bot_views = item_views[item_views.target == 1]
n_all_items = ev.item_id.nunique()
expected = n_all_items * (1 - np.exp(-len(bot_views) / n_all_items)) # столько разных было бы при случайном выборе
print(f'просмотров ботов: {len(bot_views)}, разных объявлений: {bot_views.item_id.nunique()}, при случайном выборе ~{expected:.0f}')
per_item = item_views.groupby(['item_id', 'target']).cookie_id.nunique().unstack(fill_value=0)
print('корреляция числа просмотров ботами и людьми по объявлениям:', round(per_item[0].corr(per_item[1]), 3))

bot_items = set(item_views[(item_views.day < '2026-04-13') & (item_views.target == 1)].item_id)
week2 = item_views[item_views.day >= '2026-04-13'].assign(seen_by_bots=lambda d: d.item_id.isin(bot_items)).groupby('cookie_id').agg(share=('seen_by_bots', 'mean'), target=('target', 'first'))
print('доля просмотров объявлений, которые на 1-й неделе смотрели боты:', week2.groupby('target').share.mean().round(2).to_dict())
print('ROC-AUC:', key_auc(['item_id']))

зрителей у просмотренного объявления, в среднем: {0: 2.9, 1: 4.6}
просмотров ботов: 12388, разных объявлений: 6957, при случайном выборе ~11045
корреляция числа просмотров ботами и людьми по объявлениям: -0.039
доля просмотров объявлений, которые на 1-й неделе смотрели боты: {0: 0.09, 1: 0.48}
ROC-AUC: 0.786


### Города и боты

In [52]:
pd.Series({
    'город': key_auc(['item_location']),
    'категория + город': key_auc(['item_category', 'item_location']),
    'запрос + город': key_auc(['search_query', 'item_location']),
}, name='ROC-AUC')

город                0.560
категория + город    0.572
запрос + город       0.525
Name: ROC-AUC, dtype: float64

### User-Agent и боты

In [53]:
print('ROC-AUC:', key_auc(['user_agent']))

ROC-AUC: 0.647


### Вывод

Сильно связаны с ботами только объявления: боты раз за разом смотрят один и тот же набор объявлений,
и набор с первой недели узнаётся на второй ROC-AUC 0.79. У городов 0.53-0.57, у User-Agent 0.65, и он уже есть в ua_class.

Поэтому добавляем два признака по объявлениям: сколько других кук смотрели объявление за 7 дней до конца окна и доля ботов среди кук, смотревших объявление, то есть target encoding

TE для валидации и test считается по меткам train, для train по меткам других дней.

In [54]:
def item_popularity(ev, meta, window_days=7):
    views = ev.loc[ev.item_id.notna(), ['cookie_id', 'item_id', 'day']].drop_duplicates()
    per_day = views.groupby(['day', 'item_id']).size().rename('n').reset_index()
    n_per_day = meta.groupby('window_start_ts').size()
    chunks = []
    for day in sorted(meta.window_start_ts.unique()):
        start = day - pd.Timedelta(days=window_days - 1)
        recent = per_day[(per_day.day >= start) & (per_day.day <= day)].groupby('item_id').n.sum()
        n_total = n_per_day[(n_per_day.index >= start) & (n_per_day.index <= day)].sum()
        rows = views[views.day == day].copy()
        rows['pop'] = (rows.item_id.map(recent) - 1) / n_total * 1000 # без самой куки, на 1000 кук
        chunks.append(rows)
    by_cookie = pd.concat(chunks).groupby('cookie_id')['pop']
    return pd.DataFrame({
        'item_pop_max': by_cookie.max(),
        'item_pop_mean': by_cookie.mean(),
    }).reindex(meta.cookie_id.values)


def item_bot_share(ev, target_ids, known, smooth=2.0):
    views = ev.loc[ev.item_id.notna(), ['cookie_id', 'item_id']].drop_duplicates()
    base_rate = known.target.mean()
    stats = views.merge(known[['cookie_id', 'target']], on='cookie_id').groupby('item_id').target.agg(['sum', 'count'])
    rows = views[views.cookie_id.isin(target_ids)].join(stats, on='item_id')
    rows['known'] = rows['count'].notna().astype(float) # объявление смотрела хотя бы одна размеченная кука
    rows[['sum', 'count']] = rows[['sum', 'count']].fillna(0)
    rows['rate'] = (rows['sum'] + base_rate * smooth) / (rows['count'] + smooth) # сглаженная доля ботов среди смотревших
    rows['bot_hit'] = (rows['sum'] > 0).astype(float) # объявление смотрел хотя бы один известный бот
    by_cookie = rows.groupby('cookie_id')
    return pd.DataFrame({
        'item_bot_hit_share': by_cookie.bot_hit.mean(),
        'item_bot_hit_cnt': by_cookie.bot_hit.sum(),
        'item_known_share': by_cookie.known.mean(),
        'item_bot_rate_max': by_cookie.rate.max(),
        'item_bot_rate_mean': by_cookie.rate.mean(),
    }).reindex(list(target_ids))


def encode_items(ev, known_cookies, target_ids):
    """TE для known_cookies по меткам других дней, для target_ids по всем меткам known_cookies."""
    known = known_cookies[['cookie_id', 'target', 'window_start_ts']]
    enc_target = item_bot_share(ev, target_ids, known)
    chunks = [item_bot_share(ev, group.cookie_id.values, known[known.window_start_ts != day]) for day, group in known.groupby('window_start_ts')]
    return pd.concat(chunks).reindex(known.cookie_id.values), enc_target


features = features.join(item_popularity(ev, cookies))
Xfull = features.loc[train.cookie_id]
ytr = train.target.values
print('признаков:', features.shape[1] + 5)

признаков: 93


## Валидация

In [55]:
is_valid = train.window_start_ts.ge('2026-04-17').values


def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)


def new_catboost():
    return CatBoostClassifier(iterations=800, learning_rate=0.05, depth=6, l2_leaf_reg=5, cat_features=['platform', 'ua_class'], random_seed=RANDOM_STATE, verbose=0)


# baseline из quickstart
Xbase = basic_features(ev, train)
cols = ['n_events', 'item_nunique']
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xbase.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xbase.loc[is_valid, cols])[:, 1]

# CatBoost
enc_tr, enc_va = encode_items(ev, train[~is_valid], train.cookie_id[is_valid].values)
Xtr, Xval = Xfull[~is_valid].join(enc_tr), Xfull[is_valid].join(enc_va)
p_va_cb = new_catboost().fit(Xtr, ytr[~is_valid]).predict_proba(Xval)[:, 1]

scores = pd.DataFrame({
    'P@R0.7': [precision_at_recall(ytr[is_valid], p) for p in (p_va, p_va_cb)],
    'ROC-AUC': [roc_auc_score(ytr[is_valid], p) for p in (p_va, p_va_cb)],
}, index=['baseline', 'CatBoost']).round(5)
scores

,P@R0.7,ROC-AUC
baseline,0.09572,0.57241
CatBoost,0.86923,0.95319


## Сабмит

In [56]:
enc_tr, enc_te = encode_items(ev, train, test.cookie_id.values)
Xtr = Xfull.join(enc_tr)
Xte = features.loc[test.cookie_id].join(enc_te)

model = new_catboost().fit(Xtr, ytr)
sub = pd.DataFrame({
    'cookie_id': test.cookie_id.values,
    'score': model.predict_proba(Xte[Xtr.columns])[:, 1],
})
assert len(sub) == len(test) and sub.cookie_id.is_unique and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.000303
1,ck_a76ee3b3e3e522fd,0.029320
2,ck_94c9a4d382689e82,0.026649
3,ck_8eaf9509ad9462a0,0.000209
4,ck_9a88a5a989cb5bc6,0.032106


## Важность признаков

In [57]:
pd.Series(model.get_feature_importance(), index=Xtr.columns).sort_values(ascending=False).head(15).round(2)

item_bot_rate_mean    4.38
ptr_step_mean         4.14
ptr_x_std             3.94
item_bot_hit_share    3.61
loc_per_cat           3.54
item_repeat_share     3.15
ptr_x_range           2.51
ptr_edge_share        2.41
page_mean             2.40
dtbin_30_60           2.38
dtbin_60_180          2.34
max_session_rate      2.11
dtin_mean             2.00
dtin_mad_rel          1.81
dtin_cv               1.77
dtype: float64

## Выводы
Были дубликаты, а также нужно было убрать события после окна в train. Сильнее всего работают признаки по объявлениям, разброс курсора и ритм событий. Признаки по объявлениям зависят от того, что боты смотрят тот же набор объявлений. Если сменится, модель надо переобучить.